# 4 Coupled KPOs

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import *
from tqdm import tqdm
from qutip.ui.progressbar import BaseProgressBar, TextProgressBar

## Parameters

## $\hbar = 1$

In [2]:
N = 2 # Number of KPOs
n_levels = 5 # Truncation level for Fock space
K = 1.0 # Kerr nonlinearity
p = 7.0 * K # Maximum pump strength
xi = 0.5 * K # Coupling constant
T = 700 / K # Total evolution time
num_steps = 1000 # number of time steps

## Symmetric coupling matrix J

In [3]:
J = np.random.uniform(-1, 1, (N, N))
J = (J + J.T) / 2 # Make J symmetric
np.fill_diagonal(J, 0) # Ensure no self coupling
J = Qobj(J)

In [4]:
J

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.         -0.73632783]
 [-0.73632783  0.        ]]

## Annihilation operator $\hat{a}_{i}$

In [5]:
a = [destroy(n_levels) for _ in range(N)]

In [6]:
a[0]

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=False
Qobj data =
[[0.         1.         0.         0.         0.        ]
 [0.         0.         1.41421356 0.         0.        ]
 [0.         0.         0.         1.73205081 0.        ]
 [0.         0.         0.         0.         2.        ]
 [0.         0.         0.         0.         0.        ]]

## Adiabatic evolution condition
## $\Delta_{i} = \xi_{0} \sum_{j=1}^{N} |J_{i,j}|$

In [7]:
Delta = [xi * np.sum(np.abs(J[i])) for i in range(N)]

In [8]:
Delta

[0.3681639170898287, 0.3681639170898287]

## Single KPO Hamiltonian terms

### $H_{\Delta}^{(i)} = \Delta_{i} a^\dagger_{i} a_{i}$

In [9]:
H_D = [Delta[i] * a[i].dag() * a[i] for i in range(N)]

### $H_{K}^{(i)} = \frac{K}{2} {a^\dagger_{i}}^{2} {a_{i}}^{2}$

In [10]:
H_K = [(K / 2) * a[i].dag() * a[i].dag() * a[i] * a[i] for i in range(N)]

In [11]:
H_K[0]

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=True
Qobj data =
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 3. 0.]
 [0. 0. 0. 0. 6.]]

### $H_{p}^{(i)} = {a^\dagger_{i}}^{2} + {a_{i}}^{2}$
### $\frac{p}{2}$ will be multiplied to this term during time evolution

In [12]:
H_plist = [a[i].dag() * a[i].dag() + a[i] * a[i] for i in range(N)]

In [13]:
H_plist[0]

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=True
Qobj data =
[[0.         0.         1.41421356 0.         0.        ]
 [0.         0.         0.         2.44948974 0.        ]
 [1.41421356 0.         0.         0.         3.46410162]
 [0.         2.44948974 0.         0.         0.        ]
 [0.         0.         3.46410162 0.         0.        ]]

## Coupling term
## $H_{C} = \frac{\xi_{0}}{2} \sum_{i=1}^{N} \sum_{j=1}^{N} J_{i,j} ( a_{i}^\dagger a_{j} + a_{i} a_{j}^\dagger)$ 

In [14]:
H_C = 0.5 * xi * sum([J[i, j] * (a[i].dag() * a[j] + a[i] * a[j].dag()) 
                      for i in range(N) for j in range(N)])

In [15]:
H_C

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=True
Qobj data =
[[-0.36816392  0.          0.          0.          0.        ]
 [ 0.         -1.10449175  0.          0.          0.        ]
 [ 0.          0.         -1.84081959  0.          0.        ]
 [ 0.          0.          0.         -2.57714742  0.        ]
 [ 0.          0.          0.          0.         -1.47265567]]

## Identity operator

In [16]:
I = qeye(n_levels)

In [17]:
I

Quantum object: dims=[[5], [5]], shape=(5, 5), type='oper', dtype=Dia, isherm=True
Qobj data =
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]

## Time-dependent Hamiltonian
## $H(P(t))$

In [18]:
def H_t(t, args):
    p = args['p_max'] * (t / T)
    H_p = (p / 2) * sum([tensor([H_plist[i] if i == k else I for k in range(N)]) for i in range(N)])
    H = sum([tensor([H_K[i] if i == k else I for k in range(N)]) for i in range(N)]) + \
        sum([tensor([H_D[i] if i == k else I for k in range(N)]) for i in range(N)]) - \
        tensor([H_C if i == 0 else qeye(n_levels) for i in range(N)])

    return H - H_p

## Arguments for TDH

In [19]:
H_args = {'p_max': p}

In [20]:
H_args

{'p_max': 7.0}

## Initial state $\ket{0} \otimes \ket{0} \otimes \ket{0} \otimes \ket{0}$

In [21]:
psi0 = tensor([basis(n_levels, 0) for _ in range(N)])

In [22]:
psi0

Quantum object: dims=[[5, 5], [1, 1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]

## Time evolution

In [23]:
times = np.linspace(0, T, num_steps + 1)


In [24]:
result = mesolve(H_t, psi0, times, [], [], args=H_args, options={"progress_bar": "tqdm"})

/home/aaryan-ajith-dev/Desktop/Physics/simulations/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1000/1000 [00:06<00:00, 143.45it/s]


### Extract final state

In [25]:
final_state = result.states[-1]
final_state.unit

<bound method Qobj.unit of Quantum object: dims=[[5, 5], [1, 1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[-0.16861899-0.00828826j]
 [ 0.        +0.j        ]
 [-0.32443609-0.01778486j]
 [ 0.        +0.j        ]
 [-0.23182225-0.01293497j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.28512329-0.0138337j ]
 [ 0.        +0.j        ]
 [-0.54860135-0.02972448j]
 [ 0.        +0.j        ]
 [-0.39199742-0.02162304j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.20378523-0.00986535j]
 [ 0.        +0.j        ]
 [-0.39210039-0.02120251j]
 [ 0.        +0.j        ]
 [-0.28017117-0.01542442j]]>

### The complex values arise because the system accumulates a global phase factor over the adiabatic evolution process

In [26]:
fs = Qobj(np.abs(final_state.full()))
fs

Quantum object: dims=[[25], [1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[0.16882257]
 [0.        ]
 [0.32492319]
 [0.        ]
 [0.23218284]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.28545869]
 [0.        ]
 [0.54940603]
 [0.        ]
 [0.39259335]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.20402389]
 [0.        ]
 [0.39267323]
 [0.        ]
 [0.28059544]]

## Ground state of final Hamiltonian

In [27]:
H_f = H_t(T, args=H_args)

In [28]:
H_f

Quantum object: dims=[[5, 5], [5, 5]], shape=(25, 25), type='oper', dtype=Dia, isherm=True
Qobj data =
[[ -0.27922904   0.          -4.94974747   0.           0.
    0.           0.           0.           0.           0.
   -4.94974747   0.           0.           0.           0.
    0.           0.           0.           0.           0.
    0.           0.           0.           0.           0.        ]
 [  0.           0.           0.          -8.5732141    0.
    0.           0.           0.           0.           0.
    0.          -4.94974747   0.           0.           0.
    0.           0.           0.           0.           0.
    0.           0.           0.           0.           0.        ]
 [ -4.94974747   0.           1.27922904   0.         -12.12435565
    0.           0.           0.           0.           0.
    0.           0.          -4.94974747   0.           0.
    0.           0.           0.           0.           0.
    0.           0.           0.           0.

In [29]:
ground_state = H_f.groundstate()[1]

In [30]:
gs = Qobj(np.abs(ground_state.full()))
gs

Quantum object: dims=[[25], [1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[1.35684777e-01]
 [0.00000000e+00]
 [2.66032024e-01]
 [0.00000000e+00]
 [1.91744414e-01]
 [6.93889390e-18]
 [3.46944695e-18]
 [5.20417043e-18]
 [7.52165257e-19]
 [3.08353732e-18]
 [2.90235638e-01]
 [3.55281864e-17]
 [5.69054065e-01]
 [1.97160174e-17]
 [4.10149638e-01]
 [2.37174772e-18]
 [7.23927817e-19]
 [3.49482818e-18]
 [4.81512601e-19]
 [2.32682865e-18]
 [2.08628071e-01]
 [2.03875793e-17]
 [4.09049188e-01]
 [1.36086460e-17]
 [2.94825021e-01]]

### Check if the final state after adiabatic evolution is equal to the instantaneous ground state of the final hamiltonian

In [31]:
def cossim(fs, gs):
    if np.linalg.norm(fs) * np.linalg.norm(gs) != 0:
        cossim = np.dot(fs.T, gs) / (np.linalg.norm(fs) * np.linalg.norm(gs))
        return cossim.item()  # Convert the 2D array to a scalar
    else:
        return 0  # Return 0 as a scalar

sim = np.real(cossim(fs.full(), gs.full()))
sim

0.9996828778313966

In [32]:
time_stamp = []

t1 = 0.1
t2 = 0.001

sum2 = 0
t = 0
while sum2 <= 700:
    values, _ = H_t(sum2, H_args).eigenstates()
    gaps = [values[i] - values[i - 2] for i in range(2, len(values), 2)]

    if min(gaps) < t1:
        time_stamp.append(t2)
        sum2 += t2
    else:
        time_stamp.append(t1)
        sum2+=t1
    t+=1

time_stamp = np.array(time_stamp)
np.unique(time_stamp)

KeyboardInterrupt: 

In [ ]:
sum1 = 0
prefix_sum_time = np.zeros_like(time_stamp)

for i,t in enumerate(time_stamp):
    prefix_sum_time[i] = sum1
    sum1 += t

result = mesolve(H_t, psi0, prefix_sum_time, [], [], args=H_args, options={"progress_bar": "tqdm"})
print(prefix_sum_time[-1])
result.states[-1].unit

100%|██████████| 196190/196190 [00:08<00:00, 22885.88it/s]

699.9019999989176


<bound method Qobj.unit of Quantum object: dims=[[5, 5], [1, 1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[-0.1338405 +0.00340347j]
 [ 0.        +0.j        ]
 [-0.27336796+0.00810178j]
 [ 0.        +0.j        ]
 [-0.20147471+0.00618804j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.2747022 +0.00892824j]
 [ 0.        +0.j        ]
 [-0.56108259+0.02090453j]
 [ 0.        +0.j        ]
 [-0.41348722+0.01549099j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.20224882+0.00689037j]
 [ 0.        +0.j        ]
 [-0.41305996+0.01567865j]
 [ 0.        +0.j        ]
 [-0.30444109+0.01204096j]]>

In [ ]:
final_state

<bound method Qobj.unit of Quantum object: dims=[[5, 5], [1, 1]], shape=(25, 1), type='ket', dtype=Dense
Qobj data =
[[-0.1338405 +0.00340347j]
 [ 0.        +0.j        ]
 [-0.27336796+0.00810178j]
 [ 0.        +0.j        ]
 [-0.20147471+0.00618804j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.2747022 +0.00892824j]
 [ 0.        +0.j        ]
 [-0.56108259+0.02090453j]
 [ 0.        +0.j        ]
 [-0.41348722+0.01549099j]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [-0.20224882+0.00689037j]
 [ 0.        +0.j        ]
 [-0.41305996+0.01567865j]
 [ 0.        +0.j        ]
 [-0.30444109+0.01204096j]]>